# RSX-V2-14 · Risk-gated policy calibration (T4 GPU)

48 **fresh** calibration problems × (source + finalize + remask + regenerate) = **192 planned model calls**. Preserve is free. Runs at most 12 new calls per session or 28 minutes, saves each record to Drive, and skips completed trial IDs. After all calls, fits a policy from source-only features with grouped 4-fold cross-validation. **Never uses E10 held-out outcomes for fitting.** If calibration is not better than preserve/always-finalize in the prespecified gates, confirmation is blocked by default.

Run every cell top to bottom. Re-run this notebook later to resume its own unsaved calls; never rerun old RepairScopeX experiments.

In [ ]:

import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX_E10E11'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned ALoDLM source:',CODE_COMMIT)


In [ ]:
import json,os,re,time,random,math,ast,hashlib,zipfile,zlib,base64,gc,csv,sys
from pathlib import Path
from collections import Counter,defaultdict
from types import SimpleNamespace
import numpy as np,torch,pandas as pd
from huggingface_hub import snapshot_download
from google.colab import drive,files
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('A Colab GPU (T4 or better) is required')
if torch.cuda.get_device_properties(0).total_memory<12*2**30: raise RuntimeError('At least 12 GiB GPU VRAM required')
drive.mount('/content/drive')
MODEL_ID='amazon/ALoDLM-1.7B';MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
Q=0.4;TAU=0.4;WINDOW=16
PRIOR_PROMPT='\n\nGive a short solution. End with one line: FINAL_ANSWER: <number>. For Countdown include an expression using all inputs once.'
DIRECT_PROMPT='\n\nDo not provide working. Reply with one line containing the final numerical answer. For Countdown provide a valid expression before the final number.'
FRESH_PROMPT='\n\nIndependently solve the problem from the beginning. Verify any computation. End with FINAL_ANSWER: <number>. For Countdown give a valid expression using each number once.'
def digest(v):
    return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def save_new_json(path,obj):
    if path.exists():
        old=json.loads(path.read_text());assert old==obj,'Immutable protocol/data mismatch at '+str(path)
    else:
        temp=path.with_name(path.name+'.pending');temp.write_text(json.dumps(obj,indent=2,sort_keys=True,ensure_ascii=False));os.replace(temp,path)
def atomic_record(obj):
    assert obj['protocol_hash']==PROTOCOL_HASH
    tid=obj['trial_id'];p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if p.exists():assert json.loads(p.read_text())==obj,'Conflicting trial ID '+tid
    else:
        t=p.with_suffix('.pending');t.write_text(json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False));os.replace(t,p)
def existing(tid):
    p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not p.exists():return None
    obj=json.loads(p.read_text());assert obj['trial_id']==tid and obj['protocol_hash']==PROTOCOL_HASH
    return obj
def read_records():
    out=[json.loads(f.read_text()) for f in sorted(RECS.glob('*.json'))]
    assert len(out)==len({r['trial_id'] for r in out})
    assert all(r['protocol_hash']==PROTOCOL_HASH for r in out)
    return out
def set_seed(seed):
    random.seed(seed);np.random.seed(seed%(2**32-1));torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def make_prompt(content):
    return list(tokenizer.apply_chat_template([{'role':'user','content':content}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def render(ids):return tokenizer.decode(list(map(int,ids)),skip_special_tokens=True).strip()
def model_call(prompt,prefix,mode,max_new,seed):
    set_seed(seed)
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(max_new),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats();start=time.monotonic()
    out=decoder.generate(make_prompt(prompt)+list(map(int,prefix)),cfg)
    new=list(map(int,out['token_ids']))
    return {'out':render(list(prefix)+new),'ids':list(prefix)+new,'new_ids':new,
            'generated_tokens':int(out.get('generated_tokens',len(new))),
            'loop_token_passes':int(out.get('loop_token_passes',0)),
            'outer_steps':int(out.get('outer_steps',0)),
            'stop_reason':str(out.get('stop_reason','unknown')),
            'wall_seconds':round(time.monotonic()-start,3),
            'peak_vram_gb':round(float(torch.cuda.max_memory_allocated()/2**30),3)}
def seed_for(pid):return 602600 + int(hashlib.sha256(pid.encode()).hexdigest()[:6],16) % 999999

def character_cut(ids,offset):
    for i in range(1,len(ids)):
        if len(render(ids[:i]))>=offset:return max(1,i-1)
    return max(1,len(ids)//2)
def repair_cut(ids,action):
    n=len(ids)
    if n<2:return 0
    if action=='first_math':
        m=re.search(r'(?:\d+\s*[+*/-]\s*\d+)|(?:\d+\s*=)',render(ids))
        return min(n-1,max(1,character_cut(ids,m.start()) if m else n//2))
    if action=='tail50':return min(n-1,max(1,n//2))
    raise ValueError(action)
def snapshot_export():
    archive=Path('/content')/(STUDY+'_Checkpoint.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob('*')):
            if p.is_file() and not p.name.endswith('.pending'):z.write(p,p.relative_to(ROOT))
    print('Archive',archive,'size',archive.stat().st_size,'saved records',len(read_records()))
    try:files.download(str(archive))
    except Exception as exc:print('Auto-download failed; file remains at',archive,repr(exc))
    return archive
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')

def strict_completed_correct(text,p):
    ev=grade_output(str(text).replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]',''),p)
    if p['kind']!='countdown':
        v,orig=last_explicit_number(str(text))
        return bool(closenum(v,p['target']) and orig!='conflicting_markers')
    if ev['valid_expression'] is None: return False
    # Accept only complete final-answer lines, NOT incomplete math reasoning.
    for match in re.findall(r'(?im)^\s*FINAL_ANSWER\s*:\s*(.*?)\s*$',str(text)):
        raw=match.strip().replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]','')
        try:
            if closenum(float(raw),p['target']):return True
        except (ValueError,TypeError):pass
        exp=safe_expression(raw)
        if exp and closenum(exp['value'],p['target']) and sorted(exp['literals'])==sorted(map(float,p['numbers'])):return True
    return False

def dual_score(text,p):
    main=grade_output(text,p)
    complete=strict_completed_correct(text,p)
    return {'primary':main,'complete_correct':bool(complete),
            'grading_disagreement':bool(main['success'])!=bool(complete),
            'class':('correct' if complete else ('definite_wrong' if main['eligible_semantic_failure'] else 'ambiguous'))}
# Scoring unit tests against prior observed failure modes.
test_cd={'kind':'countdown','target':14,'numbers':[2,3,4]}
assert strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: 14'.replace('\\n','\n'),test_cd)
assert not strict_completed_correct('2 + 3 * 4 = 14',test_cd)
assert not strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: \\('.replace('\\n','\n'),test_cd)


# Completed-answer rubric, label-only, may never be invoked by the router itself.
def strict_completed_correct(text,p):
    normalized=str(text).replace(r'\(','').replace(r'\)','').replace(r'\[','').replace(r'\]','')
    if p['kind']!='countdown':
        value, origin=last_explicit_number(normalized)
        return bool(closenum(value,p['target']) and origin!='conflicting_markers')
    if extract_expression(normalized,p) is None:return False
    value,origin=last_explicit_number(normalized)
    return bool(origin!='conflicting_markers' and closenum(value,p['target']))
def dual_score(text,p):
    original=grade_output(text,p)
    strict=strict_completed_correct(text,p)
    return {'primary':original,'complete_correct':strict,'grading_disagreement':bool(original['success'])!=strict,
            'class':('correct' if strict else ('definite_wrong' if original['eligible_semantic_failure'] else 'ambiguous'))}


import re
ACTIONS=['preserve','finalize','remask','regenerate']
ACTIVE=['finalize','remask','regenerate']
def bucket_of(source,kind):
    # source-only observable signature: task family, explicit final marker and stopping reason
    text=source.get('output','')
    has_marker=bool(re.search(r'(?im)^\s*FINAL_ANSWER\s*:',text) or re.search(r'\\boxed\s*\{',text))
    length_stop=source.get('stop_reason','')=='length'
    return kind+'|'+('final' if has_marker else 'no_final')+'|'+('length' if length_stop else 'eos_or_other')
def action_of(policy,source,kind):
    return policy['actions_by_bucket'].get(bucket_of(source,kind),'preserve')
# Defensive test: changing ground truth cannot change bucket or router action
mock={'output':'FINAL_ANSWER: 12','stop_reason':'eos'}
assert bucket_of(mock,'arithmetic')=='arithmetic|final|eos_or_other'
assert bucket_of(mock,'arithmetic')==bucket_of(dict(mock,target=9999),'arithmetic')

STUDY='RSX_V2_14'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/V2_14_Calibration')
RECS=ROOT/'records'; RECS.mkdir(parents=True,exist_ok=True)
MANIFEST=json.loads(zlib.decompress(base64.b64decode('eNqNVE2P2zYQ/SuEztUuSYn6yJ5aJIcCaQ9tURQoCoEih2s2EqmSlDdO0P/eob1rr712UMCQpfeGb4ZvhvxaSJWsd7F4R/4slgARwhaK70hhrJOT/bJ/DzDL+Onw9ggOgkxQ/IWfCkNG/EKFQfnVJZSpuwsibiQXDTJF3WpDmWnaljWtANNTLrUGKYQwzNDGGNkY3nQVZaCaGqqqp2Nby6oRbOz7KlegvAbMNc82JysMcC5k3wMdhQRjKsE6WkPXMt2KjjEhdGt4q8+29I58LcZVP0KWYDwXrCELhyw5gUms+BfBDUzar+nVDmTVd2yUFCpBcQ+6M6OBXgnOWFfJrm8ARqGp5vjrVK1AaybGsReK0Zr3KpcxY6JpsHqvN8sv3t1//9G///hTye7aH04RAbY2ooE5TnSdarVifd1ABbQ3uu4bLelIO9qNmjembiRuOK9erHOghxGMDzA4eEKluE4p9ziFFXJIgIOFM7g0OJ+yJ8VvGyAfGL3/wBiBz8vksYE+7IiDR+zkFiKZPf5j8zXZ8geSNjguBA1CLSRlAIJSZI3IJ0+MTcSmSAJGQLgjP8MTSThHJMG8TKgS74N0j3ml02R19p8VyBL8OMFMfnwfH46Bpbbxb29dIlIFHyNmJ3GZUPyBxJ3DOpJVe+1TFdIR6zQsgA9cOIJTm1mGT3cXY/x/Z+H5DORwtR5Gz4aYhlmmTXG+CPMFv+z2nZSfh6N632Sh6NegLvL2za20Ma16l6FfYJE2/Kr8An+Uv/PihRvSbtl3Lzf6aMbw7COelI0PKUej3+l4SBv6grx4jAN5uASeh76UwabNnMVKWpwOw2uc3cD5Dbx6je9L0f7JncufYHYd5tfhM+0Zx93GBMu59glm12F+Ha72l10K0rq3fr266t54doNj3+D4N7jqkrvw8DrFblP8NvUm14Wn1yl2m+K3qYO/Wwgv193xfsKrpuSUNyWjJa/LLR6K/wC51zlG')))
CASES=json.loads(zlib.decompress(base64.b64decode('eNrVnF1vG0l2hv9Kw1fZHdup81Wnam4Wg002CZDsRWaBXCwWi/o4NSJGJhWJGo83yH/PacqWLY1G7Ja5gnJhWCSbVOl9q57znupu/vl/Xv242fZX306vyuVmf/bO9pv26vX0anv9rtrllb/w57/4w4vLXT23d38dm+0Pdnlxudnu5/ekZkNQA+AYXNEyiabcCEpLXJOYKAcIVSkEaDzI8ogyEhfVYlVfffHRm8MoLq9+fvMTftvK+aZelv1mt/3288C+DSF8fMu7i8MAvt+d/2TT/symsTs/37334U03Q5/25erHt9Pvy3m7Pi/7m4Ourt9NuzElncq2TwhvP37abswfBiHNj68uzjeHD/9iEPPz+3L5g80v+HHzQ3t3MX/wvSPffB7um/Dqf19PX6fwIMY+hKvZKDp6LUWxKxDGQFqTv9qT8ACpAVRi7K50TL2hjC6prlcYvkphf2O/bvtZ5QgHlSHdVzkvlTkv1Bm+XmeJappL7LUqCIHkLCWULoWhDx3FMCsVzIlH14Q22Of5cCNaFUFYrzOu1PkP/tcdXu+bMezSts2m92e2nYimzZVP7bq/LG1vfRr+oRPFu7LTIs1pmeD49YLXElouMzQgdYydhoALK9E650w9gwGwNYA4skjG2GAkMJ/lVEqW9YLTSsH/afPTptukglP9MEE8zOYfNh/f9N/Xu/3Gtvu7KrMukpl1mc709TqXMLo5cJkCDUojDA2qGjFHxiQhZ5Q4sGCpSWOwUFQ4KJfaKMHg9TrzKRAt+SB3vitvXEaO+Hx8hmYFo8qoKYXcKYzQrHX1ySzVgQ0SDLLLXVunWnvNWXvgptRC6jWsl1dOxWfGmyrI9/m8uAyG5+NzDSKl95xjZa9tmdSncTMs3LWn0oOYam0OkpjyCORMyZRqiY0btJLX6xxPxWd4kM/x3tSWuEh0ic9G6GauZE7GnIGGs4M5lVol9GGRQ+8VpasfwYDQg0k3XwFYBJg4Aq6XXJ9G6DkMzYjGRYgWWCY0PBuiQ08YNNeOHcVFpVlfB4ixR6AOpL1SCjUKZSzDEe1VskjP0mjGyhMQnU6B6Bhu8l24q6+GRfpqeDZGU8TQ1UqE3j05FFOCXpKTgrwxaVXbMAy1OTcyDubIsbniI4VuHu2ewOh8KkYnutH4PqMh4jJG+4HPxuimlrzvszBSDJ4dJEKDnsxhHGu2Kt4mpgISiUXdjIqDKDef/jq8ZvbVOkM4GaPTg4yme+lOlqU7eTZG55y7Opdra5qHVbLSOWIrMsQ83wGEKB6oS/JQnSqn5MhO5P9lfzE/QXJ4YorOhxSNuATRtKwWUnw2REsvpfSYce6rRb1TrBQkQHWVNZLHkerNCUhDB0mRJqwSQmOqHth40Hqd8RSIpo/4wL/rPD4BoiO3mDo5CVoanEAgcw7o3Z5j2GOdl8LsuWPwMLahjaLnQJlDH3FgWr+RBHQqROeb1IFyV2PksKwQzgc+G6IRkrcfQEU9YfRqni6G5dgyR0kszfX3GDYz2Ugland6qEVv6nw2t5HW68wn2+bAhxCd721zRF7WIfLzbXR4K42cEUrrVQpVxK5Fqo4iWnIOMY5WraWWfTK3xjl5AS0jm7WQy/r0AfLEGB2IlzP677vT0XbX233fvd/e0xm84wRP5pgeUbwnYm8GQ8LccJgxObPFLKXMEhOmUhIFQ3+Om+c/06LasHleAQd8XKT47Qh/uSn9+08vTRfXf/vbuX07XV/Non8e/GQ/+xw+/zDtfHq/nur15vxm0t/oNiXA6b1rM33zenrzevrt6+kfD55cFF8QftyVXb2dvj/bvZ/K9JMPqvsHXlza1ZWP7OMvs9LODh5uP5n95e98O/3LJ3s//spyNf3h3/743b//9bs/fv9f//yfd83+B89H3/jYfzP91ke/yHj/Ex5x/la9+8Xj14zPryevRhgfa6O8hNRIaMPXUvAIGnJINXshjr24yXFkT0Z5eNpEdutjhRrMjxtBBwutth2W2f5p6MdMV3hRnmf3Wtx1XBbGFBbZDcvsRnLFfAKhPLb3JkRJWssZtHkmyAWYerfq2awVjwu+nEfkguDdhgajmNlzWceRsYKG1YbjMsM/D/6Y5Z50XpTnSPMCx+mNj30Z3tMy23HxKvd6AeGxJJ4GgRphyZBMPblYq9Us985jxFAhlq4YA3dxAHTNhuBgCBBDCdJWm06LV/lh6Mcsdw68KLJnX+L6m5nuC+NqXGQ4LTZ8/sBHHQdNHpKIqQF7AyDiDS/EIaPboK4x2RjQB43sTQIX8U6tRh0xDWnFAbHacV7s+M3Yj1lOEl6e5xjnYr7QdP8LTlnMD4AMMyQfS3FYfU0H7wPJEiRPxh01qWc5apjI7S6K855eMEPP0EkzjR6wl0hF+voUJyvofhj8cbrjC6R7mM3HhXTHUxZ18KYLjvFdy2jzlgqnVsRGHTUWG6m3ODRGqhZCxeydlGc3NXUaOAVGB8/3xWIrq22PC8P77eCPEt7z/UuyHXhe6OpFfSniMZ2yqB8Wixyx3Vet5/doARtXkEzIUMCLvfVccs9SRXKF1AMpJhg9kiFaoODxPaittl0XrvbbwR+zXeRlQf5mocuK0i5yytI+939zfXwswXcBKu708IVbPKcR59wqe2YHqNXJrjIw+SIPglFtvhTKC33rwx+11HG162nhYv809mOmQ35hpR3wtrYvS/D+F5y0tPNx17EKZ+7ucOZGvtyVexrqLXtThd5CbhjBWzgkKnXkSprIPOMVA0yyfq3nhWudF7oeMb+swj4T3huLbxZ67uM/abPuZdFly49VdZbBRO5vAUqVUoFAPWi0lixlTK0nX+KYNWEh65BQtLO0INVbfF5rOSzckvs09KPrXOhlOa7uePSSnpctcqFTVnSa66GvFX1skTf2Rax1INWYUxfTHi3Oe66CwXJx/1uDzrGG1ruM0hRtBCf/AC6rFzks3I27HfvR9P7CyE6HQj6Xc1y49R5OWc4RXh+0y4/uzVRf5B7OUTuCQ3y+XjZCZM9q1eOb57eSh3Hxsh6DFpqvQpzJHwKwV4PVni/dkPs09qOeywvbnUFwv+nQqeeF52zjKcs5eKGYz1o81qg3qGrkNbvQAIghWgYjqwDB07ocLnxMGAmAQ5P5X0kDC5Q2OMQyVru+cEfuduxH4R75ZTVs89b74YTLwot8Hj2bub6cw0E3fsxyadRHaDAk0RjzRfAphWTGvXXvzD3OjURFOBKFGDtAKQMi+TsCSli9JQe8fKHPQ///5rgv89nxNxOf1HBc3qnN5yUfO7mWsunwPrtwCKy9ewnXkVpmRkw5ButKRmLDu7jU3XwLRs1kUPfUx6vPtYAs79QOYz+6K5NfYKM2n1H9Zlp2gm1hZr9Xzt9dn+83V3u7WHFJe2df1uj2dW/UeiicwFd6DdgDVvUVHhBsZKgYGidubEJa67Bsi68pux3XL0+ffzddXV9cuKyu1bvpzNxHkmm721vd7X50m8Rdaz9eHa5Bncrlpeve307/6t69K9sPn4/0n97/7u7VIQv7o8c2Qm5Hfr+GPknqMu97hUw19zGGNURm9VAELVLy9SXUa1ex3KxKHJ29W6L5BFghTLRaargvdTu/rtNmu9lvyrlLflb6fF3v1X7TfvThv53+dGYfph+Kz2y6fXba724Ww3wTwbyC/PnNRdnur75w4fbgS3tXNtu7PnBauP+8yAf4ah/QmsWM8y2LJWseEtVb1R4lNzD04MpSxeMLO/iGT/4sRaJ3M2nem6qxrfYB7/vwQ7nstnWmnDk2QKb9ZflwdaNy2233ruCBdmFyLVzoG3bd/Bhn9vnxHx9/4cGnY8/3Oxf0zC5/d6/qLCQPxEU+4Ff70DQwFS0piWtUiqeKWBNpUS80Gc1JVMEsdvRkUTlLgNLdleyNRUdMq32gX6Cnne125675+bkdrjIjnlrZOmy203/str18eD0zfjvpVHc/2wFC+faIP13blR/yhQWHVzZbXzL7cn5X/rysjcu6SPyv537GUgVbFaeQB/R5YrvkqOrT3Zs4t2CU4O2bwx5yHh7jScAbdubinoCtFp+Pct9j2hfcB/4Mfl4HfsBlW+CA4ZnQn0OvTpRRoM13OQYRb5d93nvJLaKYh5eG5hVXfN4XCyoBY+ieplsikLB+qssC9OPD6E8Poh/ik9j/Zult1m8ev8/6lPzXjDGT11kRDYNLCqEUOtwn0rBib9HJw+oxqCXA+c6zUaznXnhQVKmrzYhP4788gH+SJ+IfFp4WAgjPhP8xvJmU+UpZTjp0RONQG8X50s35uwM4e6dRgMp8oUcK6k3JfPulF4wWqy+n1TboUfyzPIx/r8S3/Jcn8n9ZGM3yTPynMJBSziUPTjYc+LkzUvScP9/mV1RiSD7nh5nOkRT9Ra8WCEgYvdNbrX46yn+9k/uBPvNfV/I/xoWddXwm/pfok3YU8bYqoFdYS83n+RBWn/lE0UNnbBKoASTvtkLy4FmcPIG9s/ZstFrtvID/hA/yXx/kvzwN/8u+iOANPRP6yViT9tRqZA2HTckuBpWT+LRuhC1Zny8hrAkzaTOuKhiSMBvE9ZETwqPo/zXypwfIz/BE8idYeAn3M3Ef5+8swfm+CPI8jyVYa7VKi0JtrsTSgOfvhpj7sC5eBLwpowIsoUHPYXUbDHCU+/IrsV8+Yx/wadwHWLjLB/xcyV8yBG65IHKb7/ceKfUYMhQvv55TCdL8xQaUarZRc8gFlEINQ9lIcXXdBTxOfrpDfvhMflpJ/rCQ/OG5yO/lMnuioUwjR28DUvIC4B2Wz+/R+3y1G3FszIrogUizMM1fZaDzjUKciq5WmxaQXx8mP4SHo/8T2a8Lk78+V/D3RE/OeodM9aCjPKC5Bd57NUEFAMdQYm+Dcb6JCyBCd9Nyj94bQ2pt9cYP8KP011+hvz5Af6Qn0l+XnVVSfCb69w7eYbGMWG109V4XBaBk8FgEI8ZQ+/ztKqOhWzHEotacYJRAoDC6rPZAjtNffyX1w2f8xyfSH2nhTgQtpf9f/g8F1Lns')))
assert digest(CASES)==MANIFEST['calibration_sha256']
BANK={p['problem_id']:p for p in CASES}


PLAN=[[p['problem_id'],phase] for p in CASES for phase in ['source','finalize','remask','regenerate']]
assert len(PLAN)==len(CASES)*4
PROTOCOL={'study':STUDY,'version':MANIFEST['version'],'manifest_sha256':digest(MANIFEST),
         'cases_sha256':digest(CASES),'model_revision':MODEL_REV,'code_commit':CODE_COMMIT,
         'q':Q,'tau':TAU,'window':WINDOW,'plans':PLAN,'budgets':{k:MANIFEST[k] for k in ['source','finalize','regenerate','remask']}}
PROTOCOL_HASH=digest(PROTOCOL)
save_new_json(ROOT/'protocol.json',{'protocol_hash':PROTOCOL_HASH,'protocol':PROTOCOL})
save_new_json(ROOT/'cases.json',CASES)
print('Protocol',PROTOCOL_HASH,'existing',len(read_records()),'/',len(PLAN))


In [ ]:

sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
gc.collect();torch.cuda.empty_cache()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);tokenizer=decoder.tokenizer
assert next(decoder.model.parameters()).dtype==torch.float32
print('Pinned FP32 ALoDLM loaded')


In [ ]:

SESSION_MAX_NEW=12
SESSION_MAX_MINUTES=28
start=time.monotonic();new=0;failure=None
for pid,phase in PLAN:
    if new>=SESSION_MAX_NEW or time.monotonic()-start>=SESSION_MAX_MINUTES*60: break
    tid=f'{STUDY}|{pid}|{phase}'
    if existing(tid) is not None: continue
    p=BANK[pid];seed=seed_for(pid);src=existing(f'{STUDY}|{pid}|source')
    if phase!='source' and src is None:raise RuntimeError('Missing source: '+pid)
    try:
        if phase=='source':
            mode='left1';budget=96;prefix=[]
            res=model_call(p['prompt']+PRIOR_PROMPT,[],mode,budget,seed)
        elif phase=='finalize':
            mode='left1';budget=128;prefix=[]
            res=model_call(p['prompt']+DIRECT_PROMPT,[],mode,budget,seed)
        elif phase=='regenerate':
            mode='left1';budget=128;prefix=[]
            res=model_call(p['prompt']+FRESH_PROMPT,[],mode,budget,seed)
        else:
            mode='entropy';ids=src['token_ids']
            if len(ids)<2:
                prefix=[];budget=0
                res=dict(out=src['output'],ids=ids,new_ids=[],generated_tokens=0,loop_token_passes=0,
                         stop_reason='not_executable',wall_seconds=0.,peak_vram_gb=0.)
            else:
                cut=repair_cut(ids,'first_math');prefix=ids[:cut]
                budget=min(96,max(1,len(ids)-cut))
                res=model_call(p['prompt']+PRIOR_PROMPT,prefix,mode,budget,seed)
                assert res['ids'][:len(prefix)]==prefix
        rec={'trial_id':tid,'problem_id':pid,'kind':p['kind'],'split':p['split'],
             'phase':phase,'protocol_hash':PROTOCOL_HASH,'seed':seed,
             'output':res['out'],'token_ids':res['ids'] if phase=='source' else [],
             'stop_reason':res['stop_reason'],'generated_tokens':res['generated_tokens'],
             'loop_token_passes':res['loop_token_passes'],'wall_seconds':res['wall_seconds'],
             'peak_vram_gb':res['peak_vram_gb'],'budget':budget,'decoder':mode,
             'preserved_tokens':len(prefix) if phase=='remask' else 0,
             'score':dual_score(res['out'],p)}
        atomic_record(rec);new+=1
        print('SAVED',new,'/',SESSION_MAX_NEW,phase,pid,'outcome',rec['score']['class'],flush=True)
    except Exception as ex:
        failure=repr(ex)
        (ROOT/'last_error.json').write_text(json.dumps({'trial_id':tid,'error':failure},indent=2))
        torch.cuda.empty_cache()
        print('STOP: checkpoint preserved',failure)
        break
print('Session new:',new,'| total:',len(read_records()),'/',len(PLAN))
if failure:print('Failure details:',ROOT/'last_error.json')


In [ ]:

records=read_records(); by={(r['problem_id'],r['phase']):r for r in records}
complete=[p for p in CASES if all((p['problem_id'],a) in by for a in ['source','finalize','remask','regenerate'])]
rows=[]
for p in complete:
    pid=p['problem_id'];source=by[(pid,'source')]
    row={'problem_id':pid,'kind':p['kind'],'bucket':bucket_of(source,p['kind'])}
    for a in ['source','finalize','remask','regenerate']:
        rr=by[(pid,a)]; row[a+'_correct']=int(rr['score']['complete_correct']); row[a+'_class']=rr['score']['class']
        row[a+'_passes']=rr['loop_token_passes'];row[a+'_seconds']=rr['wall_seconds']
    rows.append(row)
df=pd.DataFrame(rows)
if len(df): display(df.groupby('kind')[[a+'_correct' for a in ['source','finalize','remask','regenerate']]].agg(['count','sum']))
print('Complete problems',len(complete),'/',len(CASES),' saved calls:',len(records))
status={'complete_problems':len(complete),'planned_problems':len(CASES),'calls':len(records),'planned_calls':len(PLAN)}
(ROOT/'progress.json').write_text(json.dumps(status,indent=2))
if len(df):df.to_csv(ROOT/'paired_summary.csv',index=False)
snapshot_export()


In [ ]:

# Fit only after the full E14 calibration grid is complete. E10/E11 outcomes are NEVER training rows.
if len(complete)!=len(CASES):
    print('NOT YET CALIBRATED — resume E14. Existing calls are preserved.')
else:
    from math import sqrt
    rows_by_pid={x['problem_id']:x for x in rows}
    thresholds=[0.0,0.05,0.10,0.15]
    min_supports=[3,5,7]
    cost_penalty_levels=[0,0.002]
    damage_caps=[0.10,0.20]
    candidates=[(t,m,c,d) for t in thresholds for m in min_supports for c in cost_penalty_levels for d in damage_caps]
    buckets=sorted(set(x['bucket'] for x in rows))
    def learn(train,hyper):
        thresh,support,cost_penalty,damage_cap=hyper
        table={}
        for b in buckets:
            group=[x for x in train if x['bucket']==b]
            n=len(group)
            if n<support:continue
            base=np.array([x['source_correct'] for x in group],dtype=float)
            opts=[]
            for a in ACTIVE:
                outcomes=np.array([x[a+'_correct'] for x in group],dtype=float)
                gain=float((outcomes-base).mean())
                harm=float(((base==1)&(outcomes==0)).mean())
                cost=float(np.mean([x[a+'_passes'] for x in group]))/2000
                utility=gain-cost_penalty*cost
                if gain>thresh and harm<=damage_cap:
                    opts.append((utility,a))
            if opts:
                opts.sort(reverse=True);table[b]=opts[0][1]
        return table
    def assess(table,items):
        chosen=[table.get(x['bucket'],'preserve') for x in items]
        corr=sum(x['source_correct'] if a=='preserve' else x[a+'_correct'] for x,a in zip(items,chosen))
        damage=sum(x['source_correct']==1 and a!='preserve' and x[a+'_correct']==0 for x,a in zip(items,chosen))
        extra=sum(0 if a=='preserve' else x[a+'_passes'] for x,a in zip(items,chosen))/len(items)
        return corr,damage,extra
    cv=[]
    for hyp in candidates:
        actual=[]
        # Group-disjoint 4-fold cross-validation on unique problem IDs
        for fold in range(4):
            tr=[x for x in rows if int(hashlib.sha256(x['problem_id'].encode()).hexdigest()[:8],16)%4!=fold]
            va=[x for x in rows if int(hashlib.sha256(x['problem_id'].encode()).hexdigest()[:8],16)%4==fold]
            if not va:continue
            table=learn(tr,hyp)
            for x in va:
                a=table.get(x['bucket'],'preserve')
                actual.append((x,x['source_correct'] if a=='preserve' else x[a+'_correct'],a))
        gain=sum(out-x['source_correct'] for x,out,_ in actual)
        damage=sum(x['source_correct']==1 and out==0 for x,out,_ in actual)
        cost=sum(0 if a=='preserve' else x[a+'_passes'] for x,_,a in actual)
        finalize_gain=sum(x['finalize_correct']-x['source_correct'] for x in rows)
        cv.append({'hyper':list(hyp),'cv_gain_vs_preserve':gain,'cv_damaged_correct':damage,
                   'cv_correct':sum(y for _,y,_ in actual),'extra_passes':cost,
                   'always_finalize_gain':finalize_gain})
    # Select conservatively: favor fewer damaged answers, then positive net gain, then computation.
    cv.sort(key=lambda v:(v['cv_gain_vs_preserve']>0,v['cv_gain_vs_preserve'], -v['cv_damaged_correct'],-v['extra_passes']),reverse=True)
    best=cv[0]; table=learn(rows,tuple(best['hyper']))
    cv_advantage=best['cv_gain_vs_preserve']-max(0,best['always_finalize_gain'])
    pass_gate=(best['cv_gain_vs_preserve']>0 and cv_advantage>=0 and best['cv_damaged_correct']<=2 and len(table)>0)
    policy={'schema_version':1,'study':'RSX_V2','training_split':'calibration',
            'calibration_cases_hash':MANIFEST['calibration_sha256'],'manifest_hash':digest(MANIFEST),
            'training_protocol_hash':PROTOCOL_HASH,'features':['kind','has_final','length_stop'],
            'actions_by_bucket':table,'hyperparameters':best['hyper'],
            'selection':'predefined grouped 4fold CV; select by positive net gain, harm and compute',
            'cross_validation':best,'calibration_gate_pass':bool(pass_gate),
            'test_set_label_access':False}
    policy['policy_sha256']=digest(policy)
    save_new_json(ROOT/'LOCKED_V2_POLICY.json',policy)
    pd.DataFrame(cv).to_csv(ROOT/'policy_cv_candidates.csv',index=False)
    print('POLICY FROZEN:',json.dumps(policy,indent=2))
    if not pass_gate:print('CALIBRATION GATE FAILED: preserve archived outcomes, do not claim method improvement.')
    else: print('CALIBRATION GATE PASSED — V2-15 may use this frozen policy.')
    snapshot_export()
